In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Hello World: วงจรควอนตัมแรกของคุณ

สร้าง [Bell state](https://en.wikipedia.org/wiki/Bell_state) (qubit ที่พัวพันกันสองตัว) แล้วรันด้วยสามวิธี:

1. **การจำลองแบบ Ideal** — ผลลัพธ์สมบูรณ์แบบ ไม่ต้องมีบัญชี
2. **การจำลองแบบ Noisy** — จำลองอุปกรณ์จริง ไม่ต้องมีบัญชี
3. **ฮาร์ดแวร์ควอนตัมจริง** — ต้องมีบัญชี IBM Quantum ฟรี (ดูขั้นตอนการตั้งค่าด้านล่าง)

## สร้าง Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## ตัวเลือกที่ 1: การจำลองแบบ Ideal (ไม่ต้องมีบัญชี)
ใช้ `StatevectorSampler` — simulator ในเครื่องที่ให้ผลลัพธ์สมบูรณ์แบบ ปราศจาก noise

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## ตัวเลือกที่ 2: การจำลองแบบ Noisy (ไม่ต้องมีบัญชี)
ใช้ `FakeManilaV2` — simulator ในเครื่องที่เลียนแบบอุปกรณ์ควอนตัม IBM จริง รวมถึงคุณสมบัติ noise ของอุปกรณ์นั้นด้วย วงจรจะต้องถูก transpile (ปรับให้เข้ากัน) กับชุด gate และการเชื่อมต่อ qubit ของอุปกรณ์ก่อน

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## ตัวเลือกที่ 3: ฮาร์ดแวร์ควอนตัมจริง
ต้องมีบัญชี IBM Quantum ฟรี หากต้องการสมัคร:

1. ลงทะเบียนที่ [quantum.cloud.ibm.com/registration](https://quantum.cloud.ibm.com/registration) — ไม่ต้องใช้บัตรเครดิตใน 30 วันแรก
2. ลงชื่อเข้าใช้ที่ [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) และเลือกภูมิภาค **us-east** (จำเป็นสำหรับ Open Plan ฟรี)
3. สร้าง instance (Open Plan ฟรี) ที่ [Instances](https://quantum.cloud.ibm.com/instances) หากยังไม่มี
4. สร้าง API key ที่ [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) (หรือที่ [cloud.ibm.com/iam/apikeys](https://cloud.ibm.com/iam/apikeys))
5. คัดลอก **CRN** (Cloud Resource Name) จากหน้า [Instances](https://quantum.cloud.ibm.com/instances) ของคุณ

หากยังไม่ได้บันทึก credentials ในเซสชัน Binder นี้ ให้รัน cell ด้านล่าง โดยแทนที่ `<your-api-key>` ด้วย API key จากขั้นตอนที่ 4 และ `<your-crn>` ด้วย CRN จากขั้นตอนที่ 5

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your-api-key>",
    instance="<your-crn>",
    set_as_default=True,
    overwrite=True,
)

**หมายเหตุ:** งานบนฮาร์ดแวร์จริงอาจใช้เวลาสักพักขึ้นอยู่กับคิวรอ หาก cell ยังรันอยู่ สามารถตรวจสอบสถานะงานและดูผลลัพธ์ได้ที่ [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## ขั้นตอนถัดไป
- **[Tutorials](https://doqumentation.org/tutorials)** — คู่มือทีละขั้นตอนเกี่ยวกับ algorithm, การลด error, transpilation และอื่น ๆ
- **[Guides](https://doqumentation.org/guides)** — เอกสารอ้างอิงเกี่ยวกับการรันวงจร, primitives และ Qiskit Runtime
- **[Courses](https://doqumentation.org/learning/courses/basics-of-quantum-information)** — เส้นทางการเรียนรู้แบบมีโครงสร้าง ตั้งแต่พื้นฐานควอนตัมไปจนถึงการคำนวณระดับ utility
- **[Modules](https://doqumentation.org/learning/modules/computer-science)** — โมดูลเชิงแนวคิดเชิงลึกด้านวิทยาการคอมพิวเตอร์และกลศาสตร์ควอนตัม